# Notebook 05: calidad de datos y descripción del conjunto

**RSNA 2022 Cervical Spine Fracture Detection — Miembro D**

Este notebook produce los tres insumos de la sección *Descripción de los datos* del informe:

1. `data/meta_train.csv` — tabla de metadatos DICOM con el **esquema completo**
   (`z_dir`, `slice_gap`, `rows`, `cols`, `z_first`, `z_last`), que el notebook 02 no
   genera porque usa una extracción más ligera.
2. `data/reporte_calidad.csv` — una fila por problema detectado, con su magnitud.
3. `data/diccionario_variables.csv` — la tabla de variables con tipo y descripción.

*Nota: debe correrse en el editor de Kaggle con el dataset de la competencia adjunto.
La extracción de metadatos recorre 2019 estudios leyendo 3 cabeceras cada uno
(sin píxeles) y tarda varios minutos; después queda cacheada.*

## 0. Preparación del entorno

In [ ]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

KAGGLE = Path("/kaggle/input").exists()
PROJECT_ROOT = Path("/kaggle/working") if KAGGLE else Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Localiza el dataset sin recorrer los ~700 mil DICOM.
dataset_candidates = [
    Path("/kaggle/input/rsna-2022-cervical-spine-fracture-detection"),
    Path("/kaggle/input/competitions/rsna-2022-cervical-spine-fracture-detection"),
    PROJECT_ROOT / "dataset",
]
if KAGGLE:
    dataset_candidates += [p.parent for p in Path("/kaggle/input").glob("*/train.csv")]
    dataset_candidates += [p.parent for p in Path("/kaggle/input").glob("*/*/train.csv")]
BASE = next((p for p in dataset_candidates
             if (p / "train.csv").is_file() and (p / "train_bounding_boxes.csv").is_file()), None)
if BASE is None:
    mounted = [p.name for p in Path("/kaggle/input").iterdir()] if KAGGLE else []
    raise FileNotFoundError(
        "No se encontró el dataset RSNA 2022. En Kaggle usa Add Input > Competition Data > "
        f"RSNA 2022 Cervical Spine Fracture Detection. Inputs montados: {mounted or 'ninguno'}"
    )

TRAIN_IMAGES = BASE / "train_images"
SEGMENTATIONS = BASE / "segmentations"
FIGURES = Path("/kaggle/working/figures") if KAGGLE else PROJECT_ROOT / "figures"
DATA_OUT = Path("/kaggle/working/data") if KAGGLE else PROJECT_ROOT / "data"
FIGURES.mkdir(parents=True, exist_ok=True)
DATA_OUT.mkdir(parents=True, exist_ok=True)

print(f"Entorno : {'Kaggle' if KAGGLE else 'local'}")
print(f"Dataset : {BASE}")
print(f"Salidas : {DATA_OUT} y {FIGURES}")

### Módulos del proyecto

`src/plots.py` y `src/quality.py` son los módulos del miembro D. La celda siguiente los
busca automáticamente en tres sitios, en orden: la raíz del proyecto, cualquier Input
montado que contenga `src/quality.py`, y —si hay Internet activado— un clon del
repositorio. Si ninguna vía funciona, falla con instrucciones concretas en lugar de
continuar con definiciones improvisadas que no coincidirían con el informe.

**Requisito:** `src/quality.py` y `src/plots.py` deben estar commiteados y subidos al
repositorio para que la vía del clon funcione.


In [ ]:
# --- Localización de los módulos del proyecto (src/) --------------------------
# En Kaggle, src/ puede llegar por tres vías. Se prueban en orden y se usa la
# primera que funcione, para no obligar a editar el notebook según el caso.
REPO_URL = "https://github.com/rodrigoajmac/CC3084_Proyecto_2"

def _tiene_src(raiz: Path) -> bool:
    return (raiz / "src" / "quality.py").is_file() and (raiz / "src" / "plots.py").is_file()

candidatos = [PROJECT_ROOT]
if KAGGLE:
    # (a) el repo subido como Input, en cualquier nivel de anidamiento
    candidatos += sorted({p.parent.parent for p in Path("/kaggle/input").glob("*/src/quality.py")})
    candidatos += sorted({p.parent.parent for p in Path("/kaggle/input").glob("*/*/src/quality.py")})
    candidatos += [Path("/kaggle/working/CC3084_Proyecto_2")]

RAIZ_SRC = next((c for c in candidatos if _tiene_src(c)), None)

# (b) último recurso: clonar el repositorio (requiere Internet activado en el notebook)
if RAIZ_SRC is None and KAGGLE:
    destino = Path("/kaggle/working/CC3084_Proyecto_2")
    if not destino.exists():
        print(f"src/ no encontrado; clonando {REPO_URL} ...")
        import subprocess
        resultado = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(destino)],
                                   capture_output=True, text=True)
        if resultado.returncode != 0:
            print(resultado.stderr[-500:])
    if _tiene_src(destino):
        RAIZ_SRC = destino

if RAIZ_SRC is None:
    _ayuda = [
        'No se encontro src/quality.py ni src/plots.py. Tres formas de resolverlo:',
        '  1. Add Input > sube el repositorio como Dataset (la via mas fiable).',
        '  2. Activa Internet en Settings y re-ejecuta: el notebook clona el repo.',
        '  3. Copia a mano la carpeta src/ a /kaggle/working/.',
        'Nota: src/quality.py y src/plots.py deben estar commiteados y subidos',
        'para que la via del clon funcione.',
    ]
    raise ModuleNotFoundError(chr(10).join(_ayuda))

if str(RAIZ_SRC) not in sys.path:
    sys.path.insert(0, str(RAIZ_SRC))
print(f"Módulos del proyecto: {RAIZ_SRC / 'src'}")

from src.plots import (aplicar_estilo, guardar, interpretar, barras_apiladas_calidad,
                       barras_cobertura, tabla_a_figura)
from src.quality import (reporte_calidad, resumen_reporte, problemas_detectados,
                         tabla_variables)
from src.metadata import construir_tabla

aplicar_estilo()

train = pd.read_csv(BASE / "train.csv")
bbox = pd.read_csv(BASE / "train_bounding_boxes.csv")
seg_paths = list(SEGMENTATIONS.glob("*.nii")) + list(SEGMENTATIONS.glob("*.nii.gz"))
seg_uids = {p.name[:-7] if p.name.endswith(".nii.gz") else p.stem for p in seg_paths}

print(f"train.csv            : {len(train):,} estudios, {train.shape[1]} variables")
print(f"bounding boxes       : {len(bbox):,} cajas sobre {bbox.StudyInstanceUID.nunique():,} estudios")
print(f"segmentaciones .nii  : {len(seg_uids):,} estudios")


## 1. Tabla de metadatos con esquema completo

El notebook 02 extrae una versión ligera de los metadatos (lee una sola cabecera por
estudio, tomada de `os.listdir()` sin ordenar). Eso basta para `slice_thickness` y
`pixel_spacing`, que son constantes dentro de un estudio, pero **no** permite calcular
`z_dir`, `slice_gap` ni el `z_extent` físico, que son justamente los chequeos de
orientación que el reporte de calidad necesita.

`src/metadata.py` lee tres cabeceras por estudio (primera, media y última, ordenadas
numéricamente) con `stop_before_pixels=True`, envuelve cada estudio en `try/except` y
cachea el resultado. Es el mismo cálculo, pero con el esquema que exige la sección 4
del informe.

In [ ]:
CACHE_META = DATA_OUT / "meta_train.csv"

meta = construir_tabla(
    train["StudyInstanceUID"].astype(str).tolist(),
    cache_path=CACHE_META,
    force_recompute=False,
)

print(f"Metadatos: {len(meta):,} estudios x {meta.shape[1]} variables")
print(f"Estudios de train.csv sin metadatos: {len(set(train.StudyInstanceUID.astype(str)) - set(meta.StudyInstanceUID.astype(str)))}")
display(meta.head())

### Nota metodológica sobre `z_extent`

Aquí `z_extent` es `|z_last - z_first|`, medido con `ImagePositionPatient[2]`: la
longitud anatómica real cubierta por el estudio. El notebook 02 la aproxima como
`n_slices x slice_thickness`, que sobreestima cuando los cortes se solapan
(reconstrucción con incremento menor al grosor), algo habitual en TC de trauma.
Las dos cifras son válidas pero no son intercambiables, y el informe usa esta.

In [ ]:
if {"z_first", "z_last", "n_slices", "slice_thickness"}.issubset(meta.columns):
    comparacion = pd.DataFrame({
        "z_extent_fisico (|z_last - z_first|)": pd.to_numeric(meta["z_extent"], errors="coerce"),
        "n_slices x slice_thickness": pd.to_numeric(meta["n_slices"], errors="coerce")
                                      * pd.to_numeric(meta["slice_thickness"], errors="coerce"),
    })
    comparacion["diferencia_mm"] = (comparacion.iloc[:, 1] - comparacion.iloc[:, 0])
    display(comparacion.describe().T.round(2))
    interpretar(
        f"La aproximación n_slices x slice_thickness difiere del recorrido físico en una "
        f"mediana de {comparacion['diferencia_mm'].median():.1f} mm. La diferencia mide el "
        "solapamiento entre cortes contiguos: cuando el incremento de reconstrucción es "
        "menor que el grosor nominal, multiplicar sobreestima la cobertura anatómica."
    )

## 2. Reporte de calidad

`reporte_calidad()` devuelve una fila por problema detectado. Cada fila trae la magnitud
absoluta, el porcentaje sobre el total, una severidad y la acción propuesta. El campo
`estado` distingue tres situaciones que conviene no confundir: `detectado` (hay un
problema real), `sin_hallazgos` (se verificó y está limpio) y `no_evaluable` (faltaba la
columna necesaria, así que la verificación no se hizo).

In [ ]:
reporte = reporte_calidad(
    train=train,
    meta=meta,
    bbox=bbox,
    seg_uids=seg_uids,
    images_dir=TRAIN_IMAGES,
)

reporte.to_csv(DATA_OUT / "reporte_calidad.csv", index=False)
print(f"Reporte guardado: {DATA_OUT / 'reporte_calidad.csv'}  ({len(reporte)} chequeos)")

pd.set_option("display.max_colwidth", 70)
display(reporte[["id", "dimension", "hallazgo", "magnitud", "unidad", "porcentaje", "severidad", "estado"]])

In [ ]:
print("--- RESUMEN POR SEVERIDAD Y ESTADO ---")
display(resumen_reporte(reporte))

detectados = problemas_detectados(reporte)
print(f"\nProblemas efectivamente detectados: {len(detectados)} de {len(reporte)} chequeos")
print(f"No evaluables (falta columna): {int((reporte.estado == 'no_evaluable').sum())}")
print(f"Verificados y limpios: {int((reporte.estado == 'sin_hallazgos').sum())}")

### Acciones propuestas

La columna `accion` es la que alimenta la lista de operaciones de limpieza y
preprocesamiento del informe. Nótese que casi ninguna acción es "eliminar filas": la
dispersión de esta tabla es variabilidad legítima de protocolo entre 12 instituciones,
no error de captura.

In [ ]:
for fila in detectados.itertuples():
    pct = "" if pd.isna(fila.porcentaje) else f" ({fila.porcentaje:.2f}%)"
    print(f"[{fila.severidad.upper():<11}] {fila.id}: {fila.hallazgo}")
    print(f"              Magnitud: {fila.magnitud} {fila.unidad}{pct}")
    print(f"              Acción  : {fila.accion}\n")

## 3. Figura del reporte de calidad

In [ ]:
fig = barras_apiladas_calidad(
    detectados.dropna(subset=["porcentaje"]),
    titulo="Problemas de calidad detectados y su magnitud",
)
guardar(fig, "05_reporte_calidad.png", carpeta=FIGURES)
plt.show()

interpretar(
    "Cada barra es un chequeo con un problema real, medido como porcentaje de estudios "
    "afectados. Las barras rojas concentran el riesgo del proyecto: la cobertura parcial "
    "de la anotación espacial y la heterogeneidad de orientación del eje z. Ninguna de "
    "las dos se resuelve descartando filas; ambas exigen una decisión explícita de "
    "preprocesamiento documentada en el informe."
)

## 4. Cobertura de la anotación espacial

In [ ]:
n_total = train["StudyInstanceUID"].nunique()
n_pos = int(train["patient_overall"].sum())
uids_bbox = set(bbox["StudyInstanceUID"].astype(str))
uids_pos = set(train.loc[train.patient_overall.eq(1), "StudyInstanceUID"].astype(str))

cobertura = {
    "Etiqueta binaria\n(train.csv)": 100.0,
    "Etiqueta por nivel\n(C1-C7)": 100.0,
    "Bounding box\n(sobre positivos)": 100 * len(uids_bbox & uids_pos) / n_pos,
    "Bounding box\n(sobre train)": 100 * len(uids_bbox) / n_total,
    "Segmentación\n(sobre train)": 100 * len(seg_uids) / n_total,
}
fig = barras_cobertura(cobertura.keys(), cobertura.values(),
                       titulo="Cobertura de cada nivel de anotación")
guardar(fig, "05_cobertura_anotacion.png", carpeta=FIGURES)
plt.show()

interpretar(
    f"La supervisión del conjunto es piramidal: los {n_total:,} estudios tienen etiqueta "
    f"binaria y por nivel, pero solo {len(uids_bbox):,} ({100*len(uids_bbox)/n_total:.2f}%) "
    f"tienen localización en 2D y {len(seg_uids):,} ({100*len(seg_uids)/n_total:.2f}%) tienen "
    "segmentación vertebral. La consecuencia práctica es que una etapa de localización "
    "supervisada dispone de menos de la cuarta parte de los casos positivos, y que la "
    "ausencia de caja jamás debe leerse como ausencia de fractura."
)

## 5. Diccionario de variables

Tabla de variables con tipo y descripción para la sección 4 del informe: las 8 etiquetas
de `train.csv` más las variables de metadatos extraídas de las cabeceras DICOM.

In [ ]:
diccionario = tabla_variables(train, meta)
diccionario.to_csv(DATA_OUT / "diccionario_variables.csv", index=False)
print(f"Diccionario guardado: {DATA_OUT / 'diccionario_variables.csv'}")
print(f"Total de variables documentadas: {len(diccionario)}")
display(diccionario)

In [ ]:
# Versión en figura para pegar en la presentación.
fig = tabla_a_figura(
    diccionario.loc[diccionario.origen.eq("train.csv"), ["variable", "tipo", "valores_unicos", "descripcion"]],
    titulo="Variables de train.csv",
)
guardar(fig, "05_tabla_variables_train.png", carpeta=FIGURES)
plt.show()

## 6. Cifras de cierre para el informe

Última celda: imprime de una sola vez todos los números que la sección de hallazgos cita,
para poder copiarlos sin volver a recorrer el notebook y sin transcribirlos a mano.

In [ ]:
niveles = [f"C{i}" for i in range(1, 8)]
conteos = train[niveles].sum()
total_fracturas = int(conteos.sum())
c2c6c7 = int(conteos[["C2", "C6", "C7"]].sum())

bbox_area = bbox.assign(area_pct=100 * bbox["width"] * bbox["height"] / (512 * 512))

print("=" * 62)
print("CIFRAS PARA EL INFORME")
print("=" * 62)
print(f"Estudios en train                   : {len(train):,}")
print(f"Positivos (patient_overall = 1)     : {int(train.patient_overall.sum()):,} "
      f"({100*train.patient_overall.mean():.2f}%)")
print(f"Total de niveles fracturados        : {total_fracturas:,}")
print(f"Concentración en C2 + C6 + C7       : {c2c6c7:,} ({100*c2c6c7/total_fracturas:.2f}%)")
print(f"Nivel más frecuente                 : {conteos.idxmax()} ({int(conteos.max())} casos)")
print(f"Nivel menos frecuente               : {conteos.idxmin()} ({int(conteos.min())} casos)")
print("-" * 62)
print(f"Cortes por estudio (mediana)        : {meta.n_slices.median():.0f} "
      f"[{int(meta.n_slices.min())}-{int(meta.n_slices.max())}]")
print(f"slice_thickness (mediana)           : {meta.slice_thickness.median():.3f} mm "
      f"[{meta.slice_thickness.min():.3f}-{meta.slice_thickness.max():.3f}]")
print(f"z_extent físico (mediana)           : {meta.z_extent.median():.1f} mm "
      f"[{meta.z_extent.min():.1f}-{meta.z_extent.max():.1f}]")
if "z_dir" in meta.columns:
    print(f"Estudios con eje z descendente      : "
          f"{int(meta.z_dir.astype(str).str.lower().eq('descendente').sum()):,}")
if "slice_gap" in meta.columns:
    print(f"Estudios con hueco de numeración    : {int(pd.to_numeric(meta.slice_gap, errors='coerce').fillna(0).gt(0).sum()):,}")
print("-" * 62)
print(f"Cajas totales                       : {len(bbox):,} sobre {len(uids_bbox):,} estudios")
print(f"Área de la caja (mediana)           : {bbox_area.area_pct.median():.3f}% del corte 512x512")
print(f"Área de la caja (percentil 95)      : {bbox_area.area_pct.quantile(.95):.3f}%")
print(f"Estudios con segmentación           : {len(seg_uids):,}")
print(f"Estudios con caja Y segmentación    : {len(seg_uids & uids_bbox):,}")
print("=" * 62)

## Conclusión del notebook

El reporte confirma que el problema dominante de este conjunto **no es suciedad de datos**
—las etiquetas están completas, sin nulos ni duplicados, y cada estudio tiene sus
imágenes— sino **heterogeneidad de adquisición y supervisión parcial**. Esa distinción
cambia la estrategia: no hay nada que limpiar en el sentido clásico, hay que normalizar
(orientación, unidades Hounsfield, espaciado isotrópico) y diseñar el modelado asumiendo
que la localización solo existe para una minoría de los casos.